In [17]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch8. attention 
</font>
- 스마트 번역기
 <br> <br>
- GNMT (Google Neural Machine Translation)
    - 인공신경망 기반 기계 번역 시스템

![attention](https://www.mdpi.com/electronics/electronics-10-01657/article_deploy/html/images/electronics-10-01657-g001-550.jpg)

# 1. pkg 불러오기 및 하이퍼 파라미터 설정

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

In [3]:
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습 데이터셋

In [4]:
raw = pd.read_csv('data/translate.csv')

In [9]:
eng_kor = raw.values.tolist()

In [10]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
k_alpha = list(set([ch for ch in korean]))
k_alpha.sort()

In [11]:
# 영어와 한글 낱말
alpha = e_alpha + k_alpha

In [12]:
# 전체 낱말 개수
alpha_total_size = len(alpha)

# 3. 숫자쌍을 가진 dict 생성
- ex) {'S' : 0, 'E' : 1, 'P' : 2,...}

In [13]:
char_to_num = {}
for i, ch in enumerate(alpha):
    char_to_num[ch] = i

In [14]:
data = eng_kor[0]

# 4. 인코더/디코더 입력, 디코더 출력
- 인코더 입력과 디코더 입력 (원-핫 인코딩), 디코더 출력

In [15]:
def encoding(eng_kor=eng_kor):
    '인코더 입력 데이터, 디코더 입력 데이터, 디코더 출력 데이터를 return'
    
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력
    
    for data in eng_kor:
        # 인코더 입력 (영어 → 숫자 → 원-핫 인코딩)
        eng = [char_to_num.get(ch) for ch in data[0]]
        eng_one = to_categorical(eng, num_classes=alpha_total_size)
        enc_in.append(eng_one)
        # 디코더 입력 ('S'+ 한글 → 숫자 → 원-핫 인코딩)
        kor = [char_to_num.get(ch) for ch in 'S'+data[1]]
        kor_one = np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        # 디코더 출력 (한글 + 'E' → 숫자)
        kor = [[char_to_num.get(ch)] for ch in data[1]+'E']
        dec_out.append(kor)
    
    # 인공신경망에 넣기 위한 전처리 : numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    
    return enc_in, dec_in, dec_out

# 5. 전체 번역데이터(독립변수, 타겟변수)

In [16]:
X_enc, X_dec, Y_dec = encoding(eng_kor)

# 6. 모델 구현

In [20]:
from tensorflow.keras.layers import Attention, Concatenate

# 인코더 입력
ENC_IN = Input(shape=(4, alpha_total_size))

# 인코더 LSTM
ENC_OUT, state_h, state_c = LSTM(
    units=MY_HIDDEN,
    return_state=True,
    return_sequences=True
)(ENC_IN)

# 디코더 입력
DEC_IN = Input(shape=(3, alpha_total_size))

# 디코더 LSTM
DEC_MID, _, _ = LSTM(
    units=MY_HIDDEN,
    return_state=True,
    return_sequences=True
)(DEC_IN,
  initial_state=[state_h, state_c])

# 어탠션 메커니즘
CONTEXT_VECTOR = Attention()([DEC_MID, ENC_OUT])

# 컨텍스트 벡터와 디코더 LSTM 출력 결합
CONTEXT_AND_LSTM = Concatenate()([CONTEXT_VECTOR, DEC_MID])

# 디코더 출력(최종 출력)
OUT = Dense(units=alpha_total_size, activation='softmax')(CONTEXT_AND_LSTM)

# 모델 구현
model = Model(
    inputs = [ENC_IN, DEC_IN],
    outputs = OUT,
    name='model1'
)

model.summary()

Model: "model1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 4, 170)]     0           []                               
                                                                                                  
 input_5 (InputLayer)           [(None, 3, 170)]     0           []                               
                                                                                                  
 lstm_3 (LSTM)                  [(None, 4, 128),     153088      ['input_4[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                             

# 7. 모델 학습

In [24]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['accuracy']
)

begin = time()

hist = model.fit(
    [X_enc, X_dec], Y_dec,
    epochs=MY_EPOCH,
    verbose=0
)

end = time()

print(f'학습시간 : {end-begin:.2f}초')

학습시간 : 33.95초


In [25]:
model.evaluate([X_enc, X_dec], Y_dec)

4/4 [==============================] - 1s 14ms/step - loss: 3.7221e-07 - accuracy: 1.0000


[3.722099677361257e-07, 1.0]

# 8. 모델 사용

In [27]:
easy_test = [['down', 'pp'],
             ['desk', 'pp'],
             ['love', 'pp'],
             ['wood', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)

In [28]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 1s 1s/step


array([[110,  66,   1],
       [148,  96,   1],
       [ 95,  65,   1],
       [ 47,  82,   1]], dtype=int64)

In [29]:
for i in range(len(pred)):
    eng = easy_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} : {kor}')

down : 아래
desk : 책상
love : 사랑
wood : 나무


In [31]:
# 오타가 있는 단어
hard_test = [['dewn', 'pp'],
             ['desj', 'pp'],
             ['loev', 'pp'],
             ['wodd', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)

In [32]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 0s 33ms/step


array([[110,  66,   1],
       [148,  96,   1],
       [ 95,  65,   1],
       [ 47,  82,   1]], dtype=int64)

In [33]:
for i in range(len(pred)):
    eng = hard_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} : {kor}')

dewn : 아래
desj : 책상
loev : 사랑
wodd : 나무
